# Robust 2-level Hadamard (smooth controls)

Stage A of the ALC pipeline: optimize a Hadamard in the pure 2-level (Pauli) Hilbert space, robust to detuning errors (V = σ_Z), with the constraint that fidelity > 0.9999.

**Why smooth control derivatives matter**: in Stage B we'll build an ALC drive proportional to du/dt of this main pulse. If du/dt is jagged (the cubic Hermite tangents wiggle hard between knots), the ALC drive will inherit those spikes. We bound `d²u/dt²` here via `R_ddu` so that du/dt is naturally clean.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
piccolo_path       = joinpath(@__DIR__, "..", "..", "..", "Piccolo.jl")
directtrajopt_path = joinpath(@__DIR__, "..", "..", "..", "DirectTrajOpt.jl")
Pkg.develop([
    Pkg.PackageSpec(path = piccolo_path),
    Pkg.PackageSpec(path = directtrajopt_path),
])
Pkg.add(["CairoMakie", "MathTeXEngine", "LaTeXStrings", "JLD2", "DataInterpolations", "Ipopt"])
Pkg.instantiate()

using Piccolo
using LinearAlgebra
using Random
using Printf
using JLD2
using CairoMakie
using MathTeXEngine
using LaTeXStrings
using DataInterpolations: CubicHermiteSpline

set_theme!(Theme(
    fonts = (;
        regular     = texfont(:text),
        bold        = texfont(:bold),
        italic      = texfont(:italic),
        bold_italic = texfont(:bolditalic),
        ticks       = "TeX Gyre Heros Makie",
    ),
    Axis = (; xgridvisible = false),
))

## 2-level Pauli system + target

In [ ]:
# Pauli operators
const σx = ComplexF64[0 1; 1 0]
const σy = ComplexF64[0 -im; im 0]
const σz = ComplexF64[1 0; 0 -1]
const I2 = Matrix{ComplexF64}(I, 2, 2)

# Drift = 0 (no anharmonicity in 2-lvl).
# Drives:  u_X·σx + u_Y·σy   (resonant single-qubit drive in rotating frame)
# Error direction for robustness:  σz   (detuning / dephasing)
const H_drift  = zeros(ComplexF64, 2, 2)
const H_drives = [σx, σy]
const H_vars   = [σz]

# Hadamard target — full 2-lvl Hilbert (no EmbeddedOperator needed)
const U_target = (1/sqrt(2)) * ComplexF64[1.0  1.0; 1.0 -1.0]

const MHz_per_radperns = 1e3 / (2π)

## Optimization knobs

In [ ]:
const T_ns       = 20.0                # gate duration (ns)
const a_bound    = 2π * 0.01           # 10 MHz drive bound per channel
const N_knots    = 30                  # ~ 1 knot per 0.7 ns — keeps Δt small
const Q_r        = 100.0               # robustness weight (push hard)
const F_THRESHOLD = 0.9999             # hard constraint
const num_iter   = 1500
const SEED       = 42

# Smoothness knobs — bound the SECOND derivative so :du (cubic-Hermite tangents)
# varies gently between knots → du/dt is a clean waveform for downstream ALC use.
const R_DDU      = 10.0                # penalty on d²u/dt²
const DDU_BOUND  = a_bound / (0.25 * T_ns)^2   # ad-hoc; tighten if du/dt still jagged

println("T_ns = $T_ns ns,  N_knots = $N_knots,  Δt = $(round(T_ns/(N_knots-1), digits=3)) ns")
println("a_bound = $(round(a_bound/(2π)*1e3, digits=2)) MHz,  Q_r = $Q_r,  F_threshold = $F_THRESHOLD")
println("R_ddu = $R_DDU,  ddu_bound = $DDU_BOUND  (smoothness of d²u/dt²)")

## Build & solve

In [ ]:
Random.seed!(SEED)

drive_bounds = fill(a_bound, length(H_drives))

# Time-independent Hamiltonian in function form (the spline template wants 2-arg closures)
H_fn      = (u, t) -> u[1] * σx + u[2] * σy
H_vars_fn = Function[(u, t) -> σz]

varsys = VariationalQuantumSystem(
    H_fn, H_vars_fn, length(H_drives), drive_bounds;
    time_dependent = true,
)

# Cubic Hermite spline pulse — random init at 10% of a_bound
times_knots   = collect(range(0.0, T_ns, length = N_knots))
controls_init = 0.1 .* a_bound .* randn(length(H_drives), N_knots)
pulse         = CubicSplinePulse(controls_init, times_knots)

qcp = VariationalSplinePulseProblem(
    varsys, pulse, U_target;
    Q              = 0.0,         # no F in objective — pushed to constraint
    Q_r            = Q_r,         # robustness against σz
    R              = 1e-3,
    R_ddu          = R_DDU,       # ← smoothness of second derivative
    du_bound       = Inf,
    ddu_bound      = DDU_BOUND,
    n_path_samples = 3,
)

# Hard fidelity constraint F ≥ 0.9999
traj_init = get_trajectory(qcp)
push!(qcp.prob.constraints,
    FinalUnitaryFidelityConstraint(U_target, :Ũ⃗, F_THRESHOLD, traj_init))

println("\nSolving (num_iter = $num_iter)...")
t0 = time()
solve!(qcp; max_iter = num_iter, print_level = 5,
    options = IpoptOptions(
        eval_hessian    = false,
        constr_viol_tol = 1e-8,
        tol             = 1e-8,
        acceptable_tol  = 1e-8,
    ))
wall = time() - t0
@printf("Solve wall time: %.1f s  (%.3f s/iter)\n", wall, wall / num_iter)

## Post-process: F, σ_Z susceptibility

In [ ]:
traj   = get_trajectory(qcp)
N_traj = size(traj[:Ũ⃗], 2)
Us     = [iso_vec_to_operator(traj[:Ũ⃗][:, k]) for k in 1:N_traj]
U_final = Us[end]

F_strict = abs2(tr(U_target' * U_final)) / 2^2
@printf("Final F_strict        = %.6f\n", F_strict)
@printf("Final infidelity J_U  = %.4e\n", 1 - F_strict)

# σ_Z susceptibility (full-Hilbert var_obj, same formula as 3-lvl version)
V_bar   = sum(U' * σz * U for U in Us) / length(Us)
var_obj = real(tr(V_bar' * V_bar)) / 2
@printf("var_obj (σ_z susc.)   = %.4e\n", var_obj)

## Controls + their derivatives (smoothness check)

If `du/dt` looks clean here, the ALC drive (Stage B) built from it will be smooth.
If it has spikes, raise `R_DDU` and re-solve.

In [ ]:
us_knots  = traj[:u]
dus_knots = traj[:du]
ts_knots  = collect(range(0.0, T_ns, length = size(us_knots, 2)))

# Reconstruct cubic Hermite splines for fine-time sampling
sp_X = CubicHermiteSpline(dus_knots[1, :], us_knots[1, :], ts_knots)
sp_Y = CubicHermiteSpline(dus_knots[2, :], us_knots[2, :], ts_knots)

ts_fine = collect(range(0.0, T_ns, length = 2000))
u_X_t   = sp_X.(ts_fine) .* MHz_per_radperns
u_Y_t   = sp_Y.(ts_fine) .* MHz_per_radperns

# Numerical derivative of the spline at the fine times (central differences)
function fd_derivative(spl, ts)
    h = 1e-4
    return [(spl(t + h) - spl(t - h)) / (2h) for t in ts]
end
dudt_X = fd_derivative(sp_X, ts_fine) .* MHz_per_radperns   # MHz / ns
dudt_Y = fd_derivative(sp_Y, ts_fine) .* MHz_per_radperns

fig = Figure(size = (1100, 700), fontsize = 18)
ax_u = Axis(fig[1, 1], ylabel = "u (MHz)",
    title = "Robust 2-lvl Hadamard — T = $(round(T_ns, digits=1)) ns")
lines!(ax_u, ts_fine, u_X_t; color = :crimson,     linewidth = 2.5, label = L"u_X")
lines!(ax_u, ts_fine, u_Y_t; color = :forestgreen, linewidth = 2.5, label = L"u_Y")
hlines!(ax_u, [a_bound, -a_bound] .* MHz_per_radperns;
    color = :gray, linestyle = :dash, linewidth = 1)
axislegend(ax_u; position = :rt)
hidexdecorations!(ax_u, grid = false)

ax_du = Axis(fig[2, 1], xlabel = "t (ns)", ylabel = "du/dt (MHz/ns)",
    title = "First derivative (the seed for the ALC drive)")
lines!(ax_du, ts_fine, dudt_X; color = :crimson,     linewidth = 2.0, label = L"du_X/dt")
lines!(ax_du, ts_fine, dudt_Y; color = :forestgreen, linewidth = 2.0, label = L"du_Y/dt")
axislegend(ax_du; position = :rt)

linkxaxes!(ax_u, ax_du)
display(fig)
fig

## Save the trajectory for downstream Stages B & C

Reuse this saved pulse to build the ALC drive (Stage B) and verify in 3-level Duffing (Stage C).

In [ ]:
outdir = joinpath(@__DIR__, "robust_2lvl_hadamard_T$(round(Int, T_ns))ns_seed$(SEED)")
mkpath(outdir)
@save joinpath(outdir, "trajectory.jld2") traj = traj
open(joinpath(outdir, "parameters.txt"), "w") do io
    println(io, "# Robust 2-lvl Hadamard — Stage A of ALC pipeline")
    println(io, "T_ns        = $T_ns")
    println(io, "N_knots     = $N_knots")
    println(io, "a_bound     = $a_bound rad/ns ($(round(a_bound/(2π)*1e3, digits=2)) MHz)")
    println(io, "Q_r         = $Q_r")
    println(io, "F_threshold = $F_THRESHOLD")
    println(io, "R_ddu       = $R_DDU")
    println(io, "ddu_bound   = $DDU_BOUND")
    println(io, "seed        = $SEED")
    println(io, "num_iter    = $num_iter")
    println(io, "wall_s      = $wall")
    println(io, "F_strict    = $F_strict")
    println(io, "var_obj     = $var_obj")
end
println("Saved to: $outdir")